# Twitter US Airline Sentiment Analysis

## 1. Install & Import Dependencies

In [ ]:
!pip install -q kagglehub nltk scikit-learn seaborn matplotlib

In [ ]:
import os
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

# Download NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

# Set plot styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## 2. Load Dataset

In [ ]:
# Download latest version from Kaggle
dataset_path = kagglehub.dataset_download("crowdflower/twitter-airline-sentiment")
csv_file = os.path.join(dataset_path, "Tweets.csv")

df = pd.read_csv(csv_file)
print(f"Dataset Shape: {df.shape}")
df.head()

## 3. Exploratory Data Analysis

In [ ]:
# 1. Overall sentiment distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

palette = {"negative": "#e74c3c", "neutral": "#95a5a6", "positive": "#2ecc71"}
sns.countplot(data=df, x='airline_sentiment', ax=axes[0], palette=palette, order=['negative', 'neutral', 'positive'])
axes[0].set_title("Distribution of Sentiment")
axes[0].set_xlabel("Sentiment")
axes[0].set_ylabel("Count")

sentiment_counts = df['airline_sentiment'].value_counts(normalize=True) * 100
axes[1].pie(sentiment_counts, labels=sentiment_counts.index, autopct='%1.1f%%', colors=['#e74c3c', '#95a5a6', '#2ecc71'], startangle=90)
axes[1].set_title("Sentiment Share (%)")

plt.tight_layout()
plt.show()

print("Class Breakdown:")
print(df['airline_sentiment'].value_counts())

In [ ]:
# 2. Sentiment breakdown by Airline
plt.figure(figsize=(12, 6))
sns.countplot(data=df, x='airline', hue='airline_sentiment', palette=palette, hue_order=['negative', 'neutral', 'positive'])
plt.title("Sentiment by Airline")
plt.xlabel("Airline")
plt.ylabel("Tweet Count")
plt.legend(title="Sentiment")
plt.show()

In [ ]:
# 3. Top Negative Reasons
plt.figure(figsize=(12, 6))
neg_reasons = df[df['airline_sentiment'] == 'negative']['negativereason'].value_counts()
sns.barplot(x=neg_reasons.values, y=neg_reasons.index, palette="viridis")
plt.title("Top Reasons for Negative Tweets")
plt.xlabel("Frequency")
plt.ylabel("Reason")
plt.show()

## 4. Text Preprocessing

In [ ]:
ps = PorterStemmer()

# Prepare stop words list, keeping critical negation words
stop_words = set(stopwords.words('english'))
negations = {'not', 'no', 'nor', 'never', 'neither', "don't", "didn't", "wasn't", "won't", "can't", "couldn't", "shouldn't"}
filtered_stopwords = stop_words - negations

def clean_tweet(text):
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # Remove Twitter mentions (@username)
    text = re.sub(r'@\w+', '', text)
    
    # Remove special characters, numbers, and punctuation
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # Tokenize
    words = nltk.word_tokenize(text)
    
    # Remove stopwords (except negations) and stem
    stemmed_words = [ps.stem(w) for w in words if w not in filtered_stopwords and len(w) > 1]
    
    return " ".join(stemmed_words)

# Apply cleaning
df['cleaned_text'] = df['text'].apply(clean_tweet)
df[['text', 'cleaned_text', 'airline_sentiment']].head(10)

## 5. Train-Test Split

In [ ]:
X = df['cleaned_text']
y = df['airline_sentiment']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training set samples: {len(X_train_raw)}")
print(f"Test set samples:     {len(X_test_raw)}")

## 6. Feature Extraction (TF-IDF)

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# Fit on train data, transform both
X_train = tfidf.fit_transform(X_train_raw)
X_test = tfidf.transform(X_test_raw)

print(f"Vocabulary size: {len(tfidf.vocabulary_)}")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

## 7. Model Training & Benchmarking

In [ ]:
models = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "Linear SVC": LinearSVC(class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

results = {}
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    weighted_f1 = f1_score(y_test, y_pred, average='weighted')
    
    results[name] = {
        "Accuracy": acc,
        "Macro F1": macro_f1,
        "Weighted F1": weighted_f1,
        "Predictions": y_pred
    }
    trained_models[name] = model

# Comparison Table
comparison_df = pd.DataFrame(results).T[['Accuracy', 'Macro F1', 'Weighted F1']].sort_values(by="Macro F1", ascending=False)
comparison_df

## 8. Model Evaluation & Confusion Matrices

In [ ]:
# Best model based on Macro F1
best_model_name = comparison_df.index[0]
best_model = trained_models[best_model_name]
best_preds = results[best_model_name]["Predictions"]

print(f"=== Classification Report: {best_model_name} ===\n")
print(classification_report(y_test, best_preds))

# Confusion Matrices
classes = ['negative', 'neutral', 'positive']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res["Predictions"], labels=classes)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=classes, yticklabels=classes)
    axes[idx].set_title(f"{name} (Acc: {res['Accuracy']:.3f})")
    axes[idx].set_xlabel("Predicted")
    axes[idx].set_ylabel("Actual")

plt.tight_layout()
plt.show()

## 9. Predict Airline Sentiment

In [ ]:
def predict_airline_sentiment(tweet_text, model=best_model, vectorizer=tfidf):
    cleaned = clean_tweet(tweet_text)
    vectorized = vectorizer.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    
    confidence_text = ""
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(vectorized)[0]
        classes = model.classes_
        conf_dict = {cls: f"{prob*100:.1f}%" for cls, prob in zip(classes, probs)}
        confidence_text = f" | Confidence: {conf_dict}"
        
    return f"Prediction: [{prediction.upper()}]{confidence_text}"

# Test cases
sample_tweets = [
    "@VirginAmerica your staff was incredible and the flight was on time! Best airline ever!",
    "@united I've been stuck on the tarmac for 3 hours with no water or updates. Terrible service.",
    "@SouthwestAir what time does flight 432 arrive at JFK?",
    "@AmericanAir not happy with the cancelled flight and lost baggage."
]

for t in sample_tweets:
    print(f"Tweet: '{t}'")
    print(predict_airline_sentiment(t))
    print("-" * 70)

## 10. Save Model

In [ ]:
import joblib

joblib.dump(best_model, "best_sentiment_model.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")
print("Model and Vectorizer saved successfully!")